In [ ]:
# groq.py  (pip install openai)
import config
from openai import OpenAI

GROQ_URL = "https://api.groq.com/openai/v1"
MODELS = getattr(config, "GROQ_MODELS", ["llama-3.1-8b-instant", "mixtral-8x7b-32768"])

def generate_response(prompt: str, temperature: float = 0.3, max_tokens: int = 512) -> str:
    key = getattr(config, "GROQ_API_KEY", None)
    if not key:
        return "Error: GROQ_API_KEY missing in config.py"
    c = OpenAI(api_key=key, base_url=GROQ_URL)

    last_err = None
    for m in MODELS:
        try:
            r = c.chat.completions.create(
                model=m,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return r.choices[0].message.content
        except Exception as e:
            last_err = e

    return (
        "Groq model failed.\n"
        f"Tried models: {MODELS}\n"
        "Fix:\n"
        "1) Switch to hf by importing hf.py in main.py OR\n"
        "2) Replace Groq model in groq.py (GROQ_MODELS).\n"
        f"Details: {type(last_err).__name__}: {last_err}"
    )

In [ ]:
import config
from huggingface_hub import InferenceClient

MODELS = getattr(
    config,
    "HF_MODELS",
    ["meta-llama/Llama-3.1-8B-Instruct"],
)

def generate_response(prompt: str, temperature: float = 0.3, max_tokens: int = 512) -> str:
    key = getattr(config, "HF_API_KEY", None)
    if not key:
        return "Error: HF_API_KEY missing in config.py"

    last_err = None
    for m in MODELS:
        try:
            c = InferenceClient(model=m, token=key)
            r = c.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return r.choices[0].message.content
        except Exception as e:
            last_err = e

    return (
        "Hugging Face model failed.\n"
        f"Tried models: {MODELS}\n"
        "Fix:\n"
        "1) Switch to Groq by importing groq.py in main.py OR\n"
        "2) Replace HF model in hf.py (HF_MODELS).\n"
        f"Details: {type(last_err).__name__}: {last_err}"
    )


In [ ]:
# Choose ONE provider by importing it:

#Change groq --> hf to use hugging face API
#Change hf --> groq to use groq API
from groq import generate_response
# from groq import generate_response

def run_activity():
    print("ZERO-SHOT, ONE-SHOT & FEW-SHOT LEARNING ACTIVITY")

    category = input("Enter a category (e.g., animal, food, city): ").strip()
    item = input(f"Enter a specific {category} to classify: ").strip()

    if not category or not item:
        print("Please fill in both fields to run the activity.")
        return

    # Zero-shot example
    zero_shot = f"Is {item} a {category}? Answer yes or no."
    print("\n--- ZERO-SHOT LEARNING ---")
    print(f"Response: {generate_response(zero_shot, temperature=0.3, max_tokens=1024)}")

    # One-shot example
    one_shot = f"""Example:
Category: fruit
Item: apple
Answer: Yes, apple is a fruit.

Now you try:
Category: {category}
Item: {item}
Answer:"""
    print("\n--- ONE-SHOT LEARNING ---")
    print(f"Response: {generate_response(one_shot, temperature=0.3, max_tokens=1024)}")

    # Few-shot example (kept same as your original prompt format)
    few_shot = f"""Example 1:
Category: fruit
Item: apple
Answer: Yes, apple is a fruit.

Now you try:
Category: {category}
Item: {item}
Answer:"""
    print("\n--- FEW-SHOT LEARNING ---")
    print(f"Response: {generate_response(few_shot, temperature=0.3, max_tokens=1024)}")

    # Creative task
    creative_prompt = f"""Write a one-sentence story about the given word.

Example 1: Word: moon
Story: The moon winked at the lovers as they shared their first kiss.

Word: {item}
Story:"""
    print("\n--- CREATIVE FEW-SHOT EXAMPLE ---")
    print(f"Response: {generate_response(creative_prompt, temperature=0.7, max_tokens=1024)}")

    # Reflection questions
    print("\n--- REFLECTION QUESTIONS ---")
    print("1. How did the responses differ between zero-shot, one-shot, and few-shot?")
    print("2. Which approach gave the most helpful response?")
    print("3. How did the examples influence the model's output?")

if __name__ == "__main__":
    run_activity()


In [ ]:
import time, csv
from colorama import Fore, Style, init

# Change this import if using Hugging Face
from groq import generate_response
# from hf import generate_response

init(autoreset=True)
history=[]# list

def ask(prompt,temp=0.3):
    start=time.time()
    ans=generate_response(prompt,temperature=temp,max_tokens=512)
    return ans, time.time()-start

def save(txt, rows):# saved open file in write mode
    with open("results.txt","w",encoding="utf-8") as f:
        f.write(txt)
    with open("comparison.csv","w",newline="",encoding="utf-8") as f:
        w=csv.writer(f)
        w.writerow(["Section","Response"])
        w.writerows(rows)

def classify(category,item):
    zero=f"Is '{item}' a '{category}'? Answer Yes/No. Give Confidence (High/Medium/Low) and one-line reason."
    one=f"""Example:
Category: Fruit
Item: Apple
Answer: Yes. Confidence: High. Reason: Apple is a fruit.

Now:
Category: {category}
Item: {item}
Answer:"""
    few=f"""Example1:
Category: Fruit
Item: Apple
Answer: Yes.

Example2:
Category: Animal
Item: Tiger
Answer: Yes.

Example3:
Category: Vehicle
Item: Car
Answer: Yes.

Now:
Category: {category}
Item: {item}
Answer with Confidence and Reason:"""
    prompts=[("Zero Shot",zero,0.3),("One Shot",one,0.3),("Few Shot",few,0.3)]
    text=""
    rows=[]
    for name,p,t in prompts:
        r,sec=ask(p,t)
        print(Fore.GREEN+f"\n{name} ({sec:.2f}s)\n"+Style.RESET_ALL+r)
        text+=f"\n{name}\n{r}\n"
        rows.append([name,r])

    creative_menu={
        "1":"Write a one sentence story about",
        "2":"Write a funny joke about",
        "3":"Write a short poem about",
        "4":"Write an inspirational quote about",
        "5":"Write a catchy advertisement for",
        "6":"Write a riddle about"
    }
    print("\nCreative Task")
    print("1.Story 2.Joke 3.Poem 4.Quote 5.Ad 6.Riddle")
    ch=input("Choice: ")
    if ch in creative_menu:
        r,_=ask(f"{creative_menu[ch]} {item}.")
        print(Fore.CYAN+r)
        text+="\nCreative\n"+r
        rows.append(["Creative",r])

    quiz,_=ask(f"Create one MCQ about {item} with 4 options and answer.")
    facts,_=ask(f"Give 3 interesting facts about {item}.")
    corr,_=ask(f"If {item} is not a {category}, tell the correct category.")
    print(Fore.YELLOW+"\nQuiz\n"+quiz)
    print(Fore.MAGENTA+"\nFacts\n"+facts)
    print(Fore.BLUE+"\nCategory Check\n"+corr)
    text+=f"\nQuiz\n{quiz}\nFacts\n{facts}\nCorrection\n{corr}"# correct category
    rows+= [["Quiz",quiz],["Facts",facts],["Correction",corr]]

    print("\nTemperature Comparison")
    for t in (0.2,0.7,1.0):
        r,_=ask(f"Describe {item} in one sentence.",t)
        print(f"T={t}: {r}")

    save(text,rows)
    print("\nSaved to results.txt and comparison.csv")

def main():
    while True:
        print("\n==== Prompt Engineering Activity ====")
        print("1. Run Activity")
        print("2. View History")
        print("3. Exit")
        c=input("Choice: ")
        if c=="1":
            cat=input("Category: ").strip()
            item=input("Item: ").strip()
            if not cat or not item:
                print("Invalid input.")
                continue
            history.append((cat,item))
            classify(cat,item)
        elif c=="2":
            print(history if history else "No history.")
        elif c=="3":
            break
        else:
            print("Invalid choice.")

if __name__=="__main__":
    main()
